# AI623 Alignment Repo Runner

Colab-first notebook for SFT, reward modeling, DPO, PPO, GRPO, RLVR, evaluation, plotting, and packaging.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

REPO_MODE = "current"  # "current" or "clone"
GITHUB_REPO_URL = "https://github.com/your-username/your-repo.git"
GITHUB_BRANCH = "main"
USE_DRIVE = False

RUN_SFT = True
RUN_RM = True
RUN_DPO = False
RUN_PPO = False
RUN_GRPO = False
RUN_RLVR = False
RUN_EVAL = True

CONFIG_PATHS = ["configs/default.yaml"]
OVERRIDES = {
    "data": {
        "hh_train_samples": 1024,
        "hh_eval_samples": 64,
        "hh_prompt_pool_samples": 256,
        "gsm_train_samples": 512,
        "gsm_eval_samples": 128,
    },
    "ppo": {"steps": 20},
    "grpo": {"steps": 20},
    "rlvr": {"steps": 30},
}

current = Path.cwd().resolve()
candidates = [current, current.parent, current / "code", current.parent / "code"]
repo_root = None

if REPO_MODE == "clone":
    workspace = Path("/content/ai623_alignment_repo")
    if not workspace.exists():
        subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)], check=True)
    repo_root = workspace / "code" if (workspace / "code").exists() else workspace
else:
    for candidate in candidates:
        if (candidate / "configs" / "default.yaml").exists():
            repo_root = candidate
            break

if repo_root is None:
    raise RuntimeError("Could not locate repo root. Set REPO_MODE='clone' or open the notebook from inside the repo.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python: {platform.python_version()}")

## Runtime Sanity Checks

This cell checks GPU visibility, VRAM, Python, and torch versions.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / 1024**3, 2))
else:
    print("Running on CPU. Training will be slow; generation and syntax checks still work.")

## Clone / Mount / Repo Setup

If you want Drive storage, set `USE_DRIVE = True` and run the cell below.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
else:
    print('Skipping Drive mount.')

## Dependency Install

In [ ]:
%pip install -q -r requirements.txt

## Hugging Face Login / Environment Variables

Set `HF_TOKEN` in Colab secrets or in the environment before loading the Llama reward/value backbones.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF login succeeded.")
    except Exception as exc:
        print(f"HF login failed: {exc}")
else:
    print("HF_TOKEN is not set. Gated Llama reward/value models will fail to load until you set it.")

## Autoreload Setup

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import gc
import json
from pprint import pprint

import pandas as pd
import torch

from data.hh_rlhf import load_hh_dataset, make_dpo_dataset, make_prompt_dataset, make_sft_dataset
from data.gsm8k import load_gsm8k_dataset
from model.loading import (
    load_policy_model,
    load_policy_tokenizer,
    load_reference_model,
    load_reward_model,
    load_reward_tokenizer,
    load_value_model,
)
from train_sft import train_sft
from train_rm import train_reward_model
from train_rl import run_dpo, run_ppo, run_grpo, run_rlvr
from eval import evaluate_candidate_vs_reference, evaluate_gsm8k_pass_at_1
from utils.config import deep_merge_dicts, load_config
from utils.io import ensure_dir, save_json
from utils.memory import format_parameter_count, get_gpu_report
from utils.plotting import plot_metric_curves

## Config Loading

In [ ]:
config = load_config(CONFIG_PATHS)
config = deep_merge_dicts(config, OVERRIDES)
config["output_dir"] = str(repo_root / "runs")
pprint(config)

## Data Sanity Checks

Print three parsed HH-RLHF examples and confirm prompt/response splitting.

In [ ]:
hh_preview = load_hh_dataset(config, config["data"]["hh_train_split"], max_samples=3)
for idx, row in enumerate(hh_preview):
    print(f"Example {idx}")
    print("PROMPT:")
    print(row["prompt"][-300:])
    print("CHOSEN:")
    print(row["chosen"][:300])
    print("REJECTED:")
    print(row["rejected"][:300])
    print("-" * 80)

## Model Loading Sanity Checks

Loads policy, reference, reward model, and value model once; prints parameter counts and memory report.

In [ ]:
policy_tok = load_policy_tokenizer(config["models"]["policy_name"])
reward_tok = load_reward_tokenizer(config["models"]["reward_name"])
policy_sanity = load_policy_model(config, trainable=True)
reference_sanity = load_reference_model(config, checkpoint=config["models"].get("sft_checkpoint"))
reward_sanity = load_reward_model(config, checkpoint=config["models"].get("rm_checkpoint"), trainable=False)
value_sanity = load_value_model(config)

print("Policy tokenizer padding:", policy_tok.padding_side, policy_tok.pad_token_id, policy_tok.eos_token_id)
print("Reward tokenizer padding:", reward_tok.padding_side, reward_tok.pad_token_id, reward_tok.eos_token_id)
print("Policy params:", format_parameter_count(policy_sanity))
print("Reference params:", format_parameter_count(reference_sanity))
print("Reward params:", format_parameter_count(reward_sanity))
print("Value params:", format_parameter_count(value_sanity))
print("GPU report:")
pprint(get_gpu_report())

del policy_sanity, reference_sanity, reward_sanity, value_sanity
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Optional SFT

In [ ]:
artifacts = {}
if RUN_SFT:
    sft_result = train_sft(config)
    artifacts["sft"] = sft_result
    config["models"]["sft_checkpoint"] = sft_result["policy_checkpoint"]
    print(sft_result)
else:
    print("Skipping SFT")

## Optional RM Training

In [ ]:
if RUN_RM:
    rm_result = train_reward_model(config)
    artifacts["rm"] = rm_result
    config["models"]["rm_checkpoint"] = rm_result["rm_checkpoint"]
    print(rm_result)
else:
    print("Skipping RM")

## Optional DPO Training

In [ ]:
if RUN_DPO:
    dpo_result = run_dpo(config)
    artifacts["dpo"] = dpo_result
    print(dpo_result)
else:
    print("Skipping DPO")

## Optional PPO Training

In [ ]:
if RUN_PPO:
    ppo_result = run_ppo(config)
    artifacts["ppo"] = ppo_result
    print(ppo_result)
else:
    print("Skipping PPO")

## Optional GRPO Training

In [ ]:
if RUN_GRPO:
    grpo_result = run_grpo(config)
    artifacts["grpo"] = grpo_result
    print(grpo_result)
else:
    print("Skipping GRPO")

## Optional RLVR Training

In [ ]:
if RUN_RLVR:
    rlvr_result = run_rlvr(config)
    artifacts["rlvr"] = rlvr_result
    print(rlvr_result)
else:
    print("Skipping RLVR")

## Evaluation

In [ ]:
eval_outputs = {}
reference_checkpoint = config["models"].get("sft_checkpoint")
if RUN_EVAL and reference_checkpoint:
    for name, meta in artifacts.items():
        if name == "rm":
            continue
        ckpt = meta.get("policy_checkpoint")
        if not ckpt:
            continue
        result = evaluate_candidate_vs_reference(config, ckpt, reference_checkpoint)
        if name == "rlvr":
            result.update(evaluate_gsm8k_pass_at_1(config, ckpt))
        out_dir = Path(ckpt) / "eval"
        ensure_dir(out_dir)
        save_json(out_dir / "eval_results.json", {k: v for k, v in result.items() if k != "sample_rows"})
        pd.DataFrame(result["sample_rows"]).to_csv(out_dir / "sample_generations.csv", index=False)
        eval_outputs[name] = {"result": result, "out_dir": str(out_dir)}
    pprint(eval_outputs)
else:
    print("Skipping evaluation because RUN_EVAL is False or no SFT reference checkpoint is available.")

## Plots / Analysis

In [ ]:
plot_paths = []
for name, meta in artifacts.items():
    run_dir = Path(meta["run_dir"])
    metrics_path = run_dir / "metrics.jsonl"
    if metrics_path.exists():
        plot_path = run_dir / "metrics.png"
        plot_metric_curves(metrics_path, plot_path)
        plot_paths.append(str(plot_path))
plot_paths

## Sample Generations Table

In [ ]:
for name, payload in eval_outputs.items():
    print(f"Samples for {name}")
    display(pd.DataFrame(payload["result"]["sample_rows"]))

## Save Outputs / Checkpoints / Metrics

In [ ]:
summary = {
    "repo_root": str(repo_root),
    "artifacts": artifacts,
    "eval_outputs": eval_outputs,
    "plots": plot_paths,
}
summary_path = repo_root / "runs" / "notebook_summary.json"
ensure_dir(summary_path.parent)
save_json(summary_path, summary)
print(summary_path)

## Zip Scripts Helper

In [ ]:
!python tools/make_scripts_zip.py
print((repo_root / 'scripts.zip').exists(), repo_root / 'scripts.zip')

## Final Summary

In [ ]:
print('Completed notebook run.')
print('Artifacts:')
pprint(artifacts)
print('Evaluation outputs:')
pprint(eval_outputs)
print('scripts.zip exists:', (repo_root / 'scripts.zip').exists())